<a href="https://colab.research.google.com/github/tutonat/detector_objetos/blob/main/Identificador_de_objetos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade ultralytics opencv-python

# **Celda 1: Instalar dependencia y cargar el modelo yolo11**

In [ ]:
# Celda 1: Instalar dependencias y cargar YOLOv11
!pip install -q ultralytics

from ultralytics import YOLO
import cv2
import numpy as np
import base64
from IPython.display import display, HTML

# Cargamos el modelo ligero de YOLOv11 (descarga automática)
model = YOLO("yolo11n.pt")
print("¡Modelo YOLOv11 cargado con éxito!")

¡Modelo YOLOv11 cargado con éxito!


# **Celda 2: Interfaz Web y Procesamiento con la Cámara del Celular**
Esta celda genera el botoncito y el reproductor de video web. Captura los fotogramas de la cámara de tu móvil, los envía a Python, YOLOv11 detecta si hay gatos o perros, dibuja el recuadro y te devuelve la imagen en tiempo real.

In [18]:
# Detección EN VIVO con la cámara (YOLO) en Google Colab
# Apunta con la cámara y verás los recuadros moviéndose sobre el video. Botón «Detener» para terminar.
# Recomendado: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU (mucho más fluido).

import base64
import time

import cv2
import numpy as np
from google.colab.output import eval_js
from IPython.display import Javascript, display
from ultralytics import YOLO

# Cargar el modelo YOLO
model = YOLO("yolo11n.pt")

# ---------------------------------------------------------------- Parte del navegador
# Muestra el video de la cámara (la TRASERA si la hay) con una capa transparente encima donde se
# dibujan los recuadros. stream_frame(texto, capa) pinta la capa que le manda Python y le devuelve
# el fotograma actual en JPEG; devuelve "" cuando se pulsa «Detener».
display(Javascript('''
var video, overlay, info, stream, stopped = false, ready = null;

async function openRearCamera() {
  try {
    const devices = await navigator.mediaDevices.enumerateDevices();
    const cams = devices.filter(d => d.kind === 'videoinput');
    let rear = null;
    for (const d of cams) {
      const l = d.label.toLowerCase();
      if (l.includes('back') || l.includes('trasera') || l.includes('environment') || l.includes('rear')) { rear = d.deviceId; break; }
    }
    // Sin etiqueta clara: en los móviles la trasera suele ser la última
    if (!rear && cams.length > 0) rear = cams[cams.length - 1].deviceId;
    return await navigator.mediaDevices.getUserMedia({
      video: rear ? { deviceId: { exact: rear } } : { facingMode: { exact: 'environment' } } });
  } catch (err) {
    try { return await navigator.mediaDevices.getUserMedia({ video: { facingMode: 'environment' } }); }
    catch (e) { return await navigator.mediaDevices.getUserMedia({ video: true }); }
  }
}

async function startCamera() {
  const box = document.createElement('div');
  box.style.cssText = 'position:relative; display:inline-block; max-width:100%;';
  video = document.createElement('video');
  video.setAttribute('playsinline', '');
  video.style.cssText = 'display:block; max-width:100%;';
  overlay = document.createElement('img');   // capa con los recuadros (PNG transparente)
  overlay.style.cssText = 'position:absolute; left:0; top:0; width:100%; height:100%; pointer-events:none;';
  const bar = document.createElement('div');
  bar.style.cssText = 'margin:6px 0; font:14px sans-serif;';
  const button = document.createElement('button');
  button.textContent = 'Detener';
  button.onclick = () => { stopped = true; };
  info = document.createElement('span');
  info.style.marginLeft = '10px';
  bar.appendChild(button);
  bar.appendChild(info);
  box.appendChild(video);
  box.appendChild(overlay);
  document.body.appendChild(bar);
  document.body.appendChild(box);
  stream = await openRearCamera();
  video.srcObject = stream;
  await video.play();
}

async function stream_frame(text, layer) {
  if (!ready) ready = startCamera();
  await ready;
  if (stopped) {
    stream.getTracks().forEach(t => t.stop());
    video.parentNode.remove();
    info.textContent = 'Detenido.';
    return '';
  }
  if (layer) overlay.src = layer;
  info.textContent = text;
  // Fotograma actual, reducido a 640 px de ancho (más rápido de enviar y de procesar)
  const w = 640, h = Math.round(video.videoHeight * 640 / video.videoWidth);
  const canvas = document.createElement('canvas');
  canvas.width = w; canvas.height = h;
  canvas.getContext('2d').drawImage(video, 0, 0, w, h);
  return canvas.toDataURL('image/jpeg', 0.8);
}
'''))


# ---------------------------------------------------------------- Parte de Python
def to_image(data_url):
    """JPEG del navegador -> imagen de OpenCV."""
    raw = base64.b64decode(data_url.split(",")[1])
    return cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)


def boxes_layer(result, height, width):
    """Capa transparente (PNG) con los recuadros y nombres de lo detectado."""
    layer = np.zeros((height, width, 4), dtype=np.uint8)
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        name = result.names[int(box.cls[0])]
        conf = float(box.conf[0])
        color = (0, 255, 0, 255)
        cv2.rectangle(layer, (x1, y1), (x2, y2), color, 2)
        label = f"{name} {conf:.0%}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(layer, (x1, max(0, y1 - th - 8)), (x1 + tw + 6, y1), color, -1)
        cv2.putText(layer, label, (x1 + 3, max(th, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0, 255), 2)
    ok, png = cv2.imencode(".png", layer)
    return "data:image/png;base64," + base64.b64encode(png).decode()


print("Permite el acceso a la cámara. Apunta y verás la detección en vivo. Pulsa «Detener» para terminar.")
layer, text = "", "Iniciando…"
last = time.time()
while True:
    data_url = eval_js(f"stream_frame({text!r}, {layer!r})")
    if not data_url:   # se pulsó «Detener»
        break
    frame = to_image(data_url)
    result = model(frame, conf=0.25, verbose=False)[0]
    layer = boxes_layer(result, frame.shape[0], frame.shape[1])
    now = time.time()
    found = sorted({result.names[int(c)] for c in result.boxes.cls}) if len(result.boxes) else []
    text = f"{1 / max(now - last, 1e-6):.1f} imágenes/s · " + (", ".join(found) if found else "nada detectado")
    last = now

print("Detección detenida.")

<IPython.core.display.Javascript object>

Permite el acceso a la cámara. Apunta y verás la detección en vivo. Pulsa «Detener» para terminar.


KeyboardInterrupt: 

probando tamaño pantalla

In [15]:
# Detección EN VIVO con la cámara (YOLO) en Google Colab
# Apunta con la cámara y verás los recuadros moviéndose sobre el video. Botón «Detener» para terminar.
# Recomendado: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU (mucho más fluido).

import base64
import time

import cv2
import numpy as np
from google.colab.output import eval_js
from IPython.display import Javascript, display
from ultralytics import YOLO

# Cargar el modelo YOLO
model = YOLO("yolov8n.pt")

# ---------------------------------------------------------------- Parte del navegador
# Muestra el video de la cámara (la TRASERA si la hay) con una capa transparente encima donde se
# dibujan los recuadros. stream_frame(texto, capa) pinta la capa que le manda Python y le devuelve
# el fotograma actual en JPEG; devuelve "" cuando se pulsa «Detener».
display(Javascript('''
var video, overlay, info, stream, stopped = false, ready = null;

async function openRearCamera() {
  try {
    const devices = await navigator.mediaDevices.enumerateDevices();
    const cams = devices.filter(d => d.kind === 'videoinput');
    let rear = null;
    for (const d of cams) {
      const l = d.label.toLowerCase();
      if (l.includes('back') || l.includes('trasera') || l.includes('environment') || l.includes('rear')) { rear = d.deviceId; break; }
    }
    // Sin etiqueta clara: en los móviles la trasera suele ser la última
    if (!rear && cams.length > 0) rear = cams[cams.length - 1].deviceId;
    return await navigator.mediaDevices.getUserMedia({
      video: rear ? { deviceId: { exact: rear } } : { facingMode: { exact: 'environment' } } });
  } catch (err) {
    try { return await navigator.mediaDevices.getUserMedia({ video: { facingMode: 'environment' } }); }
    catch (e) { return await navigator.mediaDevices.getUserMedia({ video: true }); }
  }
}

async function startCamera() {
  const box = document.createElement('div');
  // Grande: ocupa todo el ancho de la salida de la celda (los recuadros se estiran con el video)
  box.style.cssText = 'position:relative; display:block; width:100%;';
  video = document.createElement('video');
  video.setAttribute('playsinline', '');
  video.style.cssText = 'display:block; width:100%; height:auto;';
  overlay = document.createElement('img');   // capa con los recuadros (PNG transparente)
  overlay.style.cssText = 'position:absolute; left:0; top:0; width:100%; height:100%; pointer-events:none;';
  const bar = document.createElement('div');
  bar.style.cssText = 'margin:6px 0; font:16px sans-serif;';
  const button = document.createElement('button');
  button.textContent = 'Detener';
  button.onclick = () => { stopped = true; };
  info = document.createElement('span');
  info.style.marginLeft = '10px';
  bar.appendChild(button);
  bar.appendChild(info);
  box.appendChild(video);
  box.appendChild(overlay);
  document.body.appendChild(bar);
  document.body.appendChild(box);
  stream = await openRearCamera();
  video.srcObject = stream;
  await video.play();
}

async function stream_frame(text, layer) {
  if (!ready) ready = startCamera();
  await ready;
  if (stopped) {
    stream.getTracks().forEach(t => t.stop());
    video.parentNode.remove();
    info.textContent = 'Detenido.';
    return '';
  }
  if (layer) overlay.src = layer;
  info.textContent = text;
  // Fotograma actual, reducido a 640 px de ancho (más rápido de enviar y de procesar)
  const w = 640, h = Math.round(video.videoHeight * 640 / video.videoWidth);
  const canvas = document.createElement('canvas');
  canvas.width = w; canvas.height = h;
  canvas.getContext('2d').drawImage(video, 0, 0, w, h);
  return canvas.toDataURL('image/jpeg', 0.8);
}
'''))


# ---------------------------------------------------------------- Parte de Python
def to_image(data_url):
    """JPEG del navegador -> imagen de OpenCV."""
    raw = base64.b64decode(data_url.split(",")[1])
    return cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)


def boxes_layer(result, height, width):
    """Capa transparente (PNG) con los recuadros y nombres de lo detectado."""
    layer = np.zeros((height, width, 4), dtype=np.uint8)
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        name = result.names[int(box.cls[0])]
        conf = float(box.conf[0])
        color = (0, 255, 0, 255)
        cv2.rectangle(layer, (x1, y1), (x2, y2), color, 2)
        label = f"{name} {conf:.0%}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(layer, (x1, max(0, y1 - th - 8)), (x1 + tw + 6, y1), color, -1)
        cv2.putText(layer, label, (x1 + 3, max(th, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0, 255), 2)
    ok, png = cv2.imencode(".png", layer)
    return "data:image/png;base64," + base64.b64encode(png).decode()


print("Permite el acceso a la cámara. Apunta y verás la detección en vivo. Pulsa «Detener» para terminar.")
layer, text = "", "Iniciando…"
last = time.time()
while True:
    data_url = eval_js(f"stream_frame({text!r}, {layer!r})")
    if not data_url:   # se pulsó «Detener»
        break
    frame = to_image(data_url)
    result = model(frame, conf=0.25, verbose=False)[0]
    layer = boxes_layer(result, frame.shape[0], frame.shape[1])
    now = time.time()
    found = sorted({result.names[int(c)] for c in result.boxes.cls}) if len(result.boxes) else []
    text = f"{1 / max(now - last, 1e-6):.1f} imágenes/s · " + (", ".join(found) if found else "nada detectado")
    last = now

print("Detección detenida.")

<IPython.core.display.Javascript object>

Permite el acceso a la cámara. Apunta y verás la detección en vivo. Pulsa «Detener» para terminar.


MessageError: NotAllowedError: Permission denied